# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [ ]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [ ]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [ ]:
def get_embedding(text):
    result = client.models.embed_content(
        model='text-embedding-004',  # sesuaikan versi model embedding
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [ ]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [ ]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

In [ ]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-2.0-flash',  # sesuaikan versi model
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [ ]:
test_queries = [
    # TODO: isi minimal 5 query uji coba di sini
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

### Catatan Analisis Relevansi

_Tuliskan pengamatanmu di sini: query mana yang berhasil di-retrieve dengan baik, query mana yang kurang relevan hasilnya, dan bagaimana LLM merespons ketika informasi tidak tersedia di knowledge base._